**Για το προτζεκτ μας συγκεκριμένα:**
Πριν τρέξουμε τον κώδικα πρέπει να έχουμε πρόσβαση στο https://drive.google.com/drive/folders/14aAn1gU3sPOLeCUXgtLjhehMwVXI3oy5?usp=sharing και να αποθηκεύσουμε το folder Project_YOLOP στο drive μας, γιατί έχει τα απαραίτητα δεδομένα για το evaluation των data με το μοντέλο YOLOP, καθώς και τα pre-trained weights του μοντέλου.

Πρέπει πριν τρέξουμε τα παρακάτω στο colab να εχουμε διαλέξει **Runtime->Change Runtime Type** και για **hardware accelerator-> GPU (Τ4)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Πηγαίνουμε στον φάκελο content του Colab
%cd /content

# Κατεβάζουμε το YOLOP
!git clone https://github.com/hustvl/YOLOP.git

# Μπαίνουμε στον φάκελο του YOLOP
%cd YOLOP

# Εγκατάσταση εξαρτήσεων
!pip install -r requirements.txt

In [ ]:
%cd /content/YOLOP
!ls -lh weights/End-to-end.pth

In [ ]:
import os

!git checkout tools/demo.py

!sed -i 's/torch.load(opt.weights/torch.load(opt.weights[0]/' tools/demo.py
print("Η διόρθωση εφαρμόστηκε σωστά.")

# paths
weights_path = 'weights/End-to-end.pth'
source_path = '/content/drive/MyDrive/Project_YOLOP/test_media'
output_dir = 'inference/output'

# Καθαρισμός προηγούμενων αποτελεσμάτων για να μην μπερδευτούν
if os.path.exists(output_dir):
    import shutil
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

# run YOLOP
print("Ξεκινάει το Inference...")
!python tools/demo.py --source "{source_path}" --weights "{weights_path}" --device 0 --save-dir "{output_dir}"

In [ ]:
# Αντιγραφή αποτελεσμάτων στο Drive
drive_output_folder = '/content/drive/MyDrive/Project_YOLOP/Results_GPU'

if not os.path.exists(drive_output_folder):
    os.makedirs(drive_output_folder)

!cp -r inference/output/* "{drive_output_folder}"
print(f"Τα αρχεία αποθηκεύτηκαν στο: {drive_output_folder}")

Τώρα το τρέχουμε ξανά με CPU (βλέπε παρακάτω --device cpu)

In [ ]:
from google.colab import drive
import os

# Σύνδεση στο Drive
drive.mount('/content/drive')

# Κατέβασμα του YOLOP και εγκατάσταση (αφού καθάρισε η μνήμη)
%cd /content
if not os.path.exists('YOLOP'):
    !git clone https://github.com/hustvl/YOLOP.git

%cd YOLOP
!pip install -qr requirements.txt

# Κατέβασμα των βαρών (Weights) ξανά
if not os.path.exists('weights/End-to-end.pth'):
    !wget https://github.com/hustvl/YOLOP/releases/download/v1.0/End-to-end.pth -O weights/End-to-end.pth

with open('tools/demo.py', 'r') as f:
    content = f.read()
if "opt.weights[0]" not in content:
    !sed -i 's/torch.load(opt.weights/torch.load(opt.weights[0]/' tools/demo.py

# Τρέξιμο σε CPU (Benchmark)
print("Ξεκινάει το CPU Benchmark...")

source_path = '/content/drive/MyDrive/Project_YOLOP/test_media'
output_dir = 'inference/output_cpu'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --device cpu
!python tools/demo.py --source "{source_path}" --weights weights/End-to-end.pth --device cpu --save-dir "{output_dir}"

In [ ]:
import os

# Ορίζουμε τον φάκελο προορισμού στο Drive
drive_cpu_folder = '/content/drive/MyDrive/Project_YOLOP/Results_CPU'

# Τον δημιουργούμε αν δεν υπάρχει
if not os.path.exists(drive_cpu_folder):
    os.makedirs(drive_cpu_folder)

print("Μεταφορά των αποτελεσμάτων CPU στο Drive...")

# Αντιγράφουμε τα αρχεία
!cp -r inference/output_cpu/* "{drive_cpu_folder}"

print(f"Δες τον φάκελο Results_CPU στο Drive σου.")